### RAG Pilelines - Data Ingestion to Vector DB Pipeline

In [9]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
## Read all the pdf's inside the dir

def process_all_pdfs(pdf_directory):
    """process all PDF files in the dir"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf files recursively 
    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to procress.")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Processed {len(documents)} documents from {pdf_file.name}")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data/pdf")

Found 1 PDF files to procress.
Processing Statement Of Purpose.pdf
Processed 2 documents from Statement Of Purpose.pdf


In [14]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Statement Of Purpose', 'source': '../data/pdf/Statement Of Purpose.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Statement Of Purpose.pdf', 'file_type': 'pdf'}, page_content='Statement  Of  Purpose(SOP)   \nMachine  Learning  has  become  one  of  the  most  impactful  technologies  of  our  time,  transforming  \nthe\n \nway\n \ndata\n \nis\n \nanalyzed\n \nand\n \ndecisions\n \nare\n \nmade.\n \nAs\n \na\n \nComputer\n \nScience\n \nEngineering\n \nundergraduate\n \nwith\n \na\n \nCGPA\n \nof\n \n8.0,\n \nI\n \nhave\n \ndeveloped\n \na\n \nstrong\n \ninterest\n \nin\n \nMachine\n \nLearning,\n \nData\n \nScience,\n \nand\n \nData\n \nAnalytics\n \nthrough\n \nmy\n \nacademic\n \ncoursework,\n \nprojects,\n \nand\n \ncontinuous\n \nself-learning.\n \nI\n \nam\n \nexcited\n \nabout\n \nthe\n \nopportunity\n \nto\n \nparticipate\n \nin\n \nthe\n

### Text Splitting into Chunks

In [18]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")


    if split_docs:
        print(f"\n Example Chunk:")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")
    
    return split_docs

In [19]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Split 2 documents into 6 chunks.

 Example Chunk:
Content : Statement  Of  Purpose(SOP)   
Machine  Learning  has  become  one  of  the  most  impactful  technologies  of  our  time,  transforming  
the
 
way
 
data
 
is
 
analyzed
 
and
 
decisions
 
are
 
ma...
Metadata : {'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Statement Of Purpose', 'source': '../data/pdf/Statement Of Purpose.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Statement Of Purpose.pdf', 'file_type': 'pdf'}


### Embeddings & Vector DB